In [ ]:
!pip install huggingface_hub
clip_storage = "/workspaces/microns_phase3_nda/fetching_stimuli/temp_clip_storage"
dir_in_repo='stimuli/raw_microns'
repo_id='SamuelSuhai/visual_codes_age'
import datajoint as dj
import sys
import shutil
import os
sys.path.append("/workspaces/microns_phase3_nda/")
os.makedirs(clip_storage, exist_ok=True)
from microns_phase3 import nda, utils
from huggingface_hub import HfApi, login
import numpy as np
import json
login()
api = HfApi()

In [ ]:


def save_clip_subset(clip_storage, table, key_clipnums):
    for key,clip_num in key_clipnums:
        data = (table & {"condition_hash": key}).fetch1()
        clip = data["clip"]
        meta = {k:v for k,v in data.items() if k !="clip"}
        meta["fps"] = meta["fps"].item()
        meta["duration"] = float(meta["duration"])

        assert meta["fps"] == 30.0 
        assert meta["duration"] == 10.0

        np.save(os.path.join(clip_storage, f'{clip_num}.npy'), clip)
        with open(os.path.join(clip_storage, f'{clip_num}_meta.json'), 'w') as f:
            json.dump(meta, f)
        print(f"Saved clip {clip_num}")




Connecting microns@db.datajoint.com:3306


In [ ]:
restricted_movies = (nda.Clip() & [{"short_movie_name": val} for val in ['Cinematic', 'sports1m']])
all_keys_clip_numbs = [(key,clip_num) for clip_num,key in enumerate(restricted_movies.fetch("condition_hash"))]
n_clips = 100
for start_idx in range(0,len(restricted_movies),n_clips):
    stop_idx = min(len(restricted_movies), start_idx + n_clips)
    keys_clip_numbs = all_keys_clip_numbs[start_idx: stop_idx]
    save_clip_subset(clip_storage,restricted_movies, keys_clip_numbs)

    api.upload_folder(
        folder_path=clip_storage,
        repo_id=repo_id,
        repo_type='dataset',
        path_in_repo=dir_in_repo
    )
    # clear local storage before next batch
    shutil.rmtree(clip_storage)
    os.makedirs(clip_storage)
    print(f"Uploaded batch {start_idx}-{stop_idx}")